In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import random
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, losses, InputExample
from sentence_transformers.evaluation import TripletEvaluator
from torch.utils.data import DataLoader

model_name = "sentence-transformers/all-mpnet-base-v2"

labels = ['amusement', 'anger', 'awe', 'contentment',
          'disgust', 'excitement', 'fear', 'sadness']
label2id = {l: i for i, l in enumerate(labels)}

dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/datasets/ul1oinasrchr/go-emotions/goemotions_balanced.csv"
)["train"]

splits = dataset.train_test_split(test_size=0.2, seed=42)
train = splits["train"]
tmp = splits["test"].train_test_split(test_size=0.5, seed=42)
val = tmp["train"]
test = tmp["test"]

model = SentenceTransformer(model_name, device="cuda")

train_examples = [
    InputExample(texts=[row["text"]], label=label2id[row["label"]])
    for row in train
]
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=32)

train_loss = losses.BatchHardTripletLoss(model)

def make_triplets(ds, n=2000, seed=42):
    random.seed(seed)
    by_label = {}
    for row in ds:
        by_label.setdefault(row["label"], []).append(row["text"])
    labs = [l for l in by_label if len(by_label[l]) >= 2]
    a, p, neg = [], [], []
    for _ in range(n):
        lab = random.choice(labs)
        x, y = random.sample(by_label[lab], 2)
        nlab = random.choice([l for l in labs if l != lab])
        a.append(x); p.append(y); neg.append(random.choice(by_label[nlab]))
    return a, p, neg

a, p, neg = make_triplets(val)
evaluator = TripletEvaluator(anchors=a, positives=p, negatives=neg, name="emotion-val")

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    evaluator=evaluator,
    epochs=4,
    warmup_steps=100,
    evaluation_steps=500,
    output_path="/kaggle/working/emotion-embedder",
    save_best_model=True,
    show_progress_bar=True,
)

/tmp/ipykernel_58/3912438043.py:6: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, losses, InputExample
/tmp/ipykernel_58/3912438043.py:7: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers.evaluation import TripletEvaluator


Generating train split: 0 examples [00:00, ? examples/s]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss,Validation Loss,Emotion-val Cosine Accuracy
500,5.076992,No log,0.524000
1000,5.015635,No log,0.526500
1500,5.012301,No log,0.526000
1849,5.012301,No log,0.532500


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]